In [ ]:
# Descargar las capas del SIG de Asturias (paso 1: datos *raw*)

**Qué hace:** descarga capas oficiales del servidor SIG del Principado (SITPA-IDEAS) y las guarda
**tal cual**, en GeoJSON y con la fecha de descarga en el nombre. No limpia nada: eso es el paso 2.

**Por qué así:** los servicios cambian o se caen. Guardando una copia fechada, la evaluación del
TFG se puede repetir con los mismos datos y en la memoria se cita "datos descargados el día X".

**Qué descarga** (editable en `SERVICES`):
- `Sanidad/Edificios_Sanitarios`: hospitales, centros de salud, consultorios…
- `Sanidad/Farmacias_Botiquines`: farmacias y botiquines.
- `Visor/Educacion`: centros de enseñanza.

**Cómo funciona:** cada servicio ArcGIS REST tiene una lista de capas (`/layers?f=json`). Para cada
capa se piden primero los identificadores de todos sus elementos y luego los elementos por bloques,
porque el servidor devuelve como máximo unos 2.000 por consulta.

Además del GeoJSON, guarda un `manifest_<fecha>.json` con la URL, la fecha y el número de elementos
de cada capa: es la "ficha" de la descarga para la memoria.

In [ ]:
%pip install -q geopandas requests

In [ ]:
import json
import os
import re
import time
from datetime import date

import geopandas as gpd
import requests

# Base URL of the Asturias GIS server (ArcGIS REST).
BASE_URL = "https://sig.asturias.es/servicios/rest/services"

# Services to download. All feature layers of each service are downloaded.
SERVICES = [
    "Sanidad/Edificios_Sanitarios",
    "Sanidad/Farmacias_Botiquines",
    "Visor/Educacion",
]

# Output folder. In Colab with Drive mounted, e.g. "/content/drive/MyDrive/tfg/data/raw".
OUTPUT_DIR = "data/raw"

# Download date, used in file names so each download is a frozen snapshot.
TODAY = date.today().isoformat()

# Elements requested per call (below the server limit of ~2,000).
CHUNK_SIZE = 500

HEADERS = {"User-Agent": "TFG-UOC-gis-agent/0.1 (academic project)"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
## 1. Funciones de descarga

- `get_json`: hace una petición y reintenta si el servidor falla (pasa a menudo con servicios públicos).
- `list_layers`: devuelve las capas de un servicio, sin los "grupos" (carpetas que no tienen datos).
- `download_layer`: descarga todos los elementos de una capa en coordenadas WGS84 (lat/lon, EPSG:4326).

In [ ]:
def get_json(url, params=None, retries=3, wait_s=10):
    """GET a URL and return the parsed JSON, retrying on network or server errors."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=60)
            response.raise_for_status()
            data = response.json()
            # ArcGIS reports some errors with HTTP 200 and an "error" key in the body.
            if isinstance(data, dict) and "error" in data:
                raise RuntimeError(data["error"])
            return data
        except Exception as error:
            print(f"   intento {attempt}/{retries} fallido: {error}")
            if attempt == retries:
                raise
            time.sleep(wait_s)


def list_layers(service):
    """Return the feature layers (id, name, geometry type) of a MapServer service."""
    info = get_json(f"{BASE_URL}/{service}/MapServer/layers", {"f": "json"})
    return [
        {"id": layer["id"], "name": layer["name"], "geometry": layer.get("geometryType")}
        for layer in info["layers"]
        if layer.get("type") == "Feature Layer"  # skip group layers (no data of their own)
    ]


def download_layer(service, layer_id):
    """Download every feature of a layer as a GeoJSON dict (EPSG:4326)."""
    query_url = f"{BASE_URL}/{service}/MapServer/{layer_id}/query"
    # 1) Ask only for the ids: this call is not limited by the server's max record count.
    ids = get_json(query_url, {"where": "1=1", "returnIdsOnly": "true", "f": "json"})
    object_ids = sorted(ids.get("objectIds") or [])
    features = []
    # 2) Fetch the features in chunks of ids.
    for start in range(0, len(object_ids), CHUNK_SIZE):
        chunk = object_ids[start:start + CHUNK_SIZE]
        page = get_json(query_url, {
            "objectIds": ",".join(map(str, chunk)),
            "outFields": "*",
            "outSR": 4326,
            "f": "geojson",
        })
        features.extend(page.get("features", []))
    return {"type": "FeatureCollection", "features": features}, len(object_ids)


def slug(text):
    """Turn a layer name into a safe file name: 'Centros de Salud' -> 'centros_de_salud'."""
    text = text.lower()
    for accented, plain in zip("áéíóúüñ", "aeiouun"):
        text = text.replace(accented, plain)
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

In [ ]:
## 2. Ver qué capas tiene cada servicio

Antes de descargar, imprime las capas. Si algún servicio falla aquí, el servidor no es accesible
desde donde ejecutas el cuaderno (anótalo: es un riesgo de datos).

In [ ]:
layers_by_service = {}
for service in SERVICES:
    print(f"── {service}")
    try:
        layers = list_layers(service)
    except Exception as error:
        print(f"   ✗ no se pudo leer el servicio: {error}\n")
        continue
    layers_by_service[service] = layers
    for layer in layers:
        print(f"   [{layer['id']}] {layer['name']}  ({layer['geometry']})")
    print()

In [ ]:
## 3. Descargar y guardar

Cada capa se guarda como `<servicio>__<capa>_<fecha>.geojson`. Si el número de elementos descargados
no coincide con el número de identificadores, se avisa: la descarga estaría incompleta.

In [ ]:
manifest = {"download_date": TODAY, "base_url": BASE_URL, "layers": []}

for service, layers in layers_by_service.items():
    for layer in layers:
        name = f"{slug(service.split('/')[-1])}__{slug(layer['name'])}"
        path = os.path.join(OUTPUT_DIR, f"{name}_{TODAY}.geojson")
        print(f"↓ {service} / {layer['name']}")
        try:
            collection, expected = download_layer(service, layer["id"])
        except Exception as error:
            print(f"   ✗ falló: {error}\n")
            manifest["layers"].append({"service": service, "layer_id": layer["id"],
                                       "layer_name": layer["name"], "error": str(error)})
            continue
        with open(path, "w", encoding="utf-8") as f:
            json.dump(collection, f, ensure_ascii=False)
        got = len(collection["features"])
        warning = "" if got == expected else f"  ⚠ esperados {expected}"
        print(f"   ✓ {got} elementos → {path}{warning}\n")
        manifest["layers"].append({
            "service": service, "layer_id": layer["id"], "layer_name": layer["name"],
            "url": f"{BASE_URL}/{service}/MapServer/{layer['id']}",
            "features": got, "expected": expected, "file": os.path.basename(path),
        })

manifest_path = os.path.join(OUTPUT_DIR, f"manifest_{TODAY}.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("Ficha de la descarga:", manifest_path)

In [ ]:
## 4. Comprobación rápida

Abre cada fichero descargado con GeoPandas y muestra cuántos elementos tiene, sus columnas y las
dos primeras filas. Así ves qué campos trae cada capa antes de diseñar la limpieza (paso 2).

In [ ]:
for entry in manifest["layers"]:
    if "file" not in entry:
        continue
    gdf = gpd.read_file(os.path.join(OUTPUT_DIR, entry["file"]))
    print(f"── {entry['layer_name']}: {len(gdf)} elementos | geometría {gdf.geom_type.unique().tolist()}")
    print("   columnas:", [c for c in gdf.columns if c != "geometry"])
    display(gdf.drop(columns="geometry").head(2))